In [1]:
from pathlib import Path
import io
import re

import fitz
from PIL import Image
import pytesseract

import chromadb
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

c:\Users\shafw\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DOMAIN = "thesis_final_project_and_graduation"
COLLECTION_NAME = "thesis_final_project_and_graduation"

In [3]:
# =====================
# CONFIG
# =====================

BASE_DIR = Path(r"D:\code\nlp\ta-sisdas")

DOMAIN = "thesis_final_project_and_graduation"
DOMAIN_DIR = BASE_DIR / "dataset_raw" / DOMAIN
CHROMA_DIR = BASE_DIR / "chroma_db"

COLLECTION_NAME = "thesis_final_project_and_graduation"
EMBEDDING_MODEL = "bge-m3"

RESET_COLLECTION = True

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

In [4]:
# =====================
# MANUAL DOCUMENT METADATA
# =====================

DOCUMENT_METADATA_MAP = {
    "KARTU-BIMBINGAN-SKRIPSI.pdf": {
        "document_type": "thesis_form",
        "academic_year": "general",
        "document_year": "general",
        "topic": "thesis_guidance_card"
    },

    "LEMBAR-PERSETUJUAN-TEMA.pdf": {
        "document_type": "thesis_form",
        "academic_year": "general",
        "document_year": "general",
        "topic": "thesis_topic_approval"
    },

    "PEDOMAN-PENULISAN-SKRIPSI-FPSI-2022.pdf": {
        "document_type": "thesis_guide",
        "academic_year": "general",
        "document_year": "2022",
        "topic": "thesis_writing_guideline"
    },

    "PER-NO-22-TAHUN-2018-YUDISIUM-DAN-WISUDA-UNIVERSITAS-NEGERI-MALANG.pdf": {
        "document_type": "graduation_regulation",
        "academic_year": "general",
        "document_year": "2018",
        "topic": "yudisium_and_graduation"
    },

    "PERTOR-NO-44-TAHUN-2022-PERUBAHAN-KEDUA-ATAS-PERATURAN-REKTOR-UNIVERSITAS-NEGERI-MALANG-NOMOR-24-TAHUN-2020-TENTANG-PEDOMAN-PENDIDIKAN-EDISI-2020-1.pdf": {
        "document_type": "academic_guide_revision",
        "academic_year": "general",
        "document_year": "2022",
        "topic": "graduation_requirement"
    },

    "SOP-Pembimbingan-Skripsi-Sah.pdf": {
        "document_type": "thesis_sop",
        "academic_year": "general",
        "document_year": "general",
        "topic": "thesis_supervision"
    },

    "SURAT-EDARAN-WISUDA-2025.pdf": {
        "document_type": "graduation_announcement",
        "academic_year": "general",
        "document_year": "2025",
        "topic": "graduation_ceremony"
    },

    "Surat_Dinas-wisuda-132-Tahun-2025.pdf": {
        "document_type": "graduation_letter",
        "academic_year": "general",
        "document_year": "2025",
        "topic": "graduation_ceremony"
    }
}


def get_document_metadata(pdf_path: Path) -> dict:
    file_name = pdf_path.name

    if file_name not in DOCUMENT_METADATA_MAP:
        raise ValueError(
            f"Metadata untuk file ini belum diset manual: {file_name}"
        )

    return DOCUMENT_METADATA_MAP[file_name]

In [5]:
# =====================
# HELPER
# =====================

def make_safe_id(text: str) -> str:
    text = text.replace(" ", "_")
    text = re.sub(r"[^a-zA-Z0-9_\-]", "_", text)
    text = re.sub(r"_+", "_", text)
    return text.strip("_")


def load_pdf_normal(pdf_path: Path, domain: str):
    loader = PyPDFLoader(str(pdf_path))
    docs = loader.load()

    cleaned_docs = []
    extra_metadata = get_document_metadata(pdf_path)

    for doc in docs:
        text = doc.page_content.strip()

        if not text:
            continue

        page = doc.metadata.get("page", 0)

        try:
            page = int(page) + 1
        except Exception:
            page = None

        doc.metadata = {
            "domain": domain,
            "source": str(pdf_path),
            "file_name": pdf_path.name,
            "page": page,
            "extraction_method": "pypdf",
            **extra_metadata
        }

        cleaned_docs.append(doc)

    return cleaned_docs


def load_pdf_ocr(pdf_path: Path, domain: str):
    pdf = fitz.open(str(pdf_path))
    docs = []
    extra_metadata = get_document_metadata(pdf_path)

    for page_number, page in enumerate(pdf, start=1):
        pix = page.get_pixmap(matrix=fitz.Matrix(3, 3), alpha=False)
        image = Image.open(io.BytesIO(pix.tobytes("png")))

        text = pytesseract.image_to_string(image, lang="ind+eng").strip()

        print(f"  OCR page {page_number}: {len(text)} chars")

        if text:
            docs.append(
                Document(
                    page_content=text,
                    metadata={
                        "domain": domain,
                        "source": str(pdf_path),
                        "file_name": pdf_path.name,
                        "page": page_number,
                        "extraction_method": "ocr_tesseract_ind_eng",
                        **extra_metadata
                    }
                )
            )

    pdf.close()
    return docs


def load_pdf_smart(pdf_path: Path, domain: str, min_chars: int = 100):
    print(f"\nLoading: {pdf_path.name}")

    normal_docs = load_pdf_normal(pdf_path, domain)
    normal_chars = sum(len(doc.page_content) for doc in normal_docs)

    print(f"  Normal extraction chars: {normal_chars}")

    if normal_chars >= min_chars:
        print("  Using normal extraction")
        return normal_docs

    print("  Normal extraction too small. Using OCR")
    return load_pdf_ocr(pdf_path, domain)

In [6]:
# =====================
# PREPARE CHROMA
# =====================

embeddings = OllamaEmbeddings(
    model=EMBEDDING_MODEL
)

test_vector = embeddings.embed_query("tes embedding")
print("Embedding dimension:", len(test_vector))

if len(test_vector) == 0:
    raise ValueError("Embedding gagal. Pastikan Ollama jalan dan model bge-m3 sudah di-pull.")


client = chromadb.PersistentClient(path=str(CHROMA_DIR))

if RESET_COLLECTION:
    try:
        client.delete_collection(name=COLLECTION_NAME)
        print(f"Old collection deleted: {COLLECTION_NAME}")
    except Exception:
        print(f"No old collection found: {COLLECTION_NAME}")


vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(CHROMA_DIR)
)

Embedding dimension: 1024
No old collection found: thesis_final_project_and_graduation


In [7]:
# =====================
# TEXT SPLITTER
# =====================

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500,
    chunk_overlap=100
)

In [8]:
# =====================
# INDEX ALL PDF IN DOMAIN
# =====================

pdf_files = sorted(DOMAIN_DIR.glob("*.pdf"))

print(f"\nTotal PDF files found: {len(pdf_files)}")

total_docs = 0
total_chunks = 0
failed_files = []

for pdf_path in pdf_files:
    try:
        docs = load_pdf_smart(pdf_path, DOMAIN)

        print(f"  Total docs/pages loaded: {len(docs)}")

        if len(docs) == 0:
            print("  Skipped: no text extracted")
            failed_files.append((pdf_path.name, "No text extracted"))
            continue

        splits = text_splitter.split_documents(docs)

        splits = [
            split for split in splits
            if split.page_content and split.page_content.strip()
        ]

        file_key = make_safe_id(pdf_path.stem)

        for i, split in enumerate(splits):
            split.metadata["chunk_index"] = i
            split.metadata["file_key"] = file_key

        ids = [
            f"{DOMAIN}-{file_key}-page-{doc.metadata.get('page')}-chunk-{doc.metadata.get('chunk_index')}"
            for doc in splits
        ]

        print(f"  Total chunks: {len(splits)}")

        if len(splits) == 0:
            print("  Skipped: no chunks created")
            failed_files.append((pdf_path.name, "No chunks created"))
            continue

        vectorstore.add_documents(
            documents=splits,
            ids=ids
        )

        total_docs += len(docs)
        total_chunks += len(splits)

        print(f"  Indexed successfully: {pdf_path.name}")

    except Exception as e:
        print(f"  Failed: {pdf_path.name}")
        print(f"  Error: {e}")
        failed_files.append((pdf_path.name, str(e)))


Total PDF files found: 8

Loading: KARTU-BIMBINGAN-SKRIPSI.pdf
  Normal extraction chars: 695
  Using normal extraction
  Total docs/pages loaded: 1
  Total chunks: 1
  Indexed successfully: KARTU-BIMBINGAN-SKRIPSI.pdf

Loading: LEMBAR-PERSETUJUAN-TEMA.pdf
  Normal extraction chars: 3209
  Using normal extraction
  Total docs/pages loaded: 1
  Total chunks: 1
  Indexed successfully: LEMBAR-PERSETUJUAN-TEMA.pdf

Loading: PEDOMAN-PENULISAN-SKRIPSI-FPSI-2022.pdf
  Normal extraction chars: 56808
  Using normal extraction
  Total docs/pages loaded: 43
  Total chunks: 72
  Indexed successfully: PEDOMAN-PENULISAN-SKRIPSI-FPSI-2022.pdf

Loading: PER-NO-22-TAHUN-2018-YUDISIUM-DAN-WISUDA-UNIVERSITAS-NEGERI-MALANG.pdf
  Normal extraction chars: 0
  Normal extraction too small. Using OCR
  OCR page 1: 1579 chars
  OCR page 2: 1796 chars
  OCR page 3: 2068 chars
  OCR page 4: 2232 chars
  OCR page 5: 408 chars
  Total docs/pages loaded: 5
  Total chunks: 9
  Indexed successfully: PER-NO-22-TAHUN-2

In [9]:
# =====================
# SUMMARY
# =====================

print("\n" + "=" * 100)
print("INDEXING SUMMARY")
print("=" * 100)

print("Domain:", DOMAIN)
print("Collection:", COLLECTION_NAME)
print("Total PDFs:", len(pdf_files))
print("Total docs/pages:", total_docs)
print("Total chunks indexed:", total_chunks)
print("Total data in collection:", vectorstore._collection.count())

if failed_files:
    print("\nFailed files:")
    for file_name, reason in failed_files:
        print("-", file_name, "=>", reason)
else:
    print("\nNo failed files.")


INDEXING SUMMARY
Domain: thesis_final_project_and_graduation
Collection: thesis_final_project_and_graduation
Total PDFs: 8
Total docs/pages: 69
Total chunks indexed: 117
Total data in collection: 117

No failed files.


In [10]:
from collections import defaultdict

collection = client.get_collection(name=COLLECTION_NAME)

data = collection.get(include=["metadatas"])

metadata_values = defaultdict(set)

for metadata in data["metadatas"]:
    for key, value in metadata.items():
        metadata_values[key].add(value)

for key, values in metadata_values.items():
    print("=" * 80)
    print("Metadata key:", key)
    print("Values:")
    for value in sorted(values, key=lambda x: str(x)):
        print("-", value)

Metadata key: topic
Values:
- graduation_ceremony
- graduation_requirement
- thesis_guidance_card
- thesis_supervision
- thesis_topic_approval
- thesis_writing_guideline
- yudisium_and_graduation
Metadata key: academic_year
Values:
- general
Metadata key: chunk_index
Values:
- 0
- 1
- 10
- 11
- 12
- 13
- 14
- 15
- 16
- 17
- 18
- 19
- 2
- 20
- 21
- 22
- 23
- 24
- 25
- 26
- 27
- 28
- 29
- 3
- 30
- 31
- 32
- 33
- 34
- 35
- 36
- 37
- 38
- 39
- 4
- 40
- 41
- 42
- 43
- 44
- 45
- 46
- 47
- 48
- 49
- 5
- 50
- 51
- 52
- 53
- 54
- 55
- 56
- 57
- 58
- 59
- 6
- 60
- 61
- 62
- 63
- 64
- 65
- 66
- 67
- 68
- 69
- 7
- 70
- 71
- 8
- 9
Metadata key: document_type
Values:
- academic_guide_revision
- graduation_announcement
- graduation_letter
- graduation_regulation
- thesis_form
- thesis_guide
- thesis_sop
Metadata key: source
Values:
- D:\code\nlp\ta-sisdas\dataset_raw\thesis_final_project_and_graduation\KARTU-BIMBINGAN-SKRIPSI.pdf
- D:\code\nlp\ta-sisdas\dataset_raw\thesis_final_project_and_graduation

In [11]:
from collections import Counter

data = collection.get(include=["metadatas"])

counter = Counter(
    metadata.get("file_name", "UNKNOWN")
    for metadata in data["metadatas"]
)

for file_name, count in counter.items():
    print(file_name, ":", count, "chunks")

KARTU-BIMBINGAN-SKRIPSI.pdf : 1 chunks
LEMBAR-PERSETUJUAN-TEMA.pdf : 1 chunks
PEDOMAN-PENULISAN-SKRIPSI-FPSI-2022.pdf : 72 chunks
PER-NO-22-TAHUN-2018-YUDISIUM-DAN-WISUDA-UNIVERSITAS-NEGERI-MALANG.pdf : 9 chunks
PERTOR-NO-44-TAHUN-2022-PERUBAHAN-KEDUA-ATAS-PERATURAN-REKTOR-UNIVERSITAS-NEGERI-MALANG-NOMOR-24-TAHUN-2020-TENTANG-PEDOMAN-PENDIDIKAN-EDISI-2020-1.pdf : 11 chunks
SOP-Pembimbingan-Skripsi-Sah.pdf : 13 chunks
SURAT-EDARAN-WISUDA-2025.pdf : 6 chunks
Surat_Dinas-wisuda-132-Tahun-2025.pdf : 4 chunks
